# Privacy & Governance Assessment
### NovaCred Credit Application Dataset

This notebook analyzes privacy risks in the NovaCred credit application dataset.

Objectives:
- Identify Personally Identifiable Information (PII)
- Demonstrate pseudonymization / anonymization
- Evaluate GDPR compliance risks
- Propose governance controls


## 1. Dataset Overview

This dataset contains credit application records used by NovaCred's machine learning system.

Each record includes:
- Applicant personal information
- Financial attributes
- Behavioral spending patterns
- Loan approval decision

Because this data is used for automated credit decisions, it falls under GDPR and EU AI Act regulatory scrutiny.



In [2]:
import json
from pathlib import Path
import pandas as pd

DATA_PATH = Path("../data/raw_credit_applications.json")

with open(DATA_PATH, "r", encoding="utf-8") as f:
    raw = json.load(f)

type(raw), len(raw)

(list, 502)

In [3]:
# Peek at first record structure
first = raw[0]
print("Top-level keys:", list(first.keys()))
print("\nExample nested keys (applicant_info):", list(first.get("applicant_info", {}).keys()))
print("Example nested keys (financials):", list(first.get("financials", {}).keys()))
print("Example nested keys (decision):", list(first.get("decision", {}).keys()))

Top-level keys: ['_id', 'applicant_info', 'financials', 'spending_behavior', 'decision', 'processing_timestamp']

Example nested keys (applicant_info): ['full_name', 'email', 'ssn', 'ip_address', 'gender', 'date_of_birth', 'zip_code']
Example nested keys (financials): ['annual_income', 'credit_history_months', 'debt_to_income', 'savings_balance']
Example nested keys (decision): ['loan_approved', 'rejection_reason']


In [4]:
# Flatten nested JSON -> tabular dataframe for auditing
df = pd.json_normalize(raw, sep=".")
df.shape

(502, 21)

In [5]:
pd.set_option("display.max_columns", 200)
pd.set_option("display.max_colwidth", 80)

df.head(3)

,_id,spending_behavior,processing_timestamp,applicant_info.full_name,applicant_info.email,applicant_info.ssn,applicant_info.ip_address,applicant_info.gender,applicant_info.date_of_birth,applicant_info.zip_code,financials.annual_income,financials.credit_history_months,financials.debt_to_income,financials.savings_balance,decision.loan_approved,decision.rejection_reason,loan_purpose,decision.interest_rate,decision.approved_amount,financials.annual_salary,notes
0,app_200,"[{'category': 'Shopping', 'amount': 480}, {'category': 'Rent', 'amount': 790...",2024-01-15T00:00:00Z,Jerry Smith,jerry.smith17@hotmail.com,596-64-4340,192.168.48.155,Male,2001-03-09,10036,73000,23,0.20,31212,False,algorithm_risk_score,NaN,NaN,NaN,NaN,NaN
1,app_037,"[{'category': 'Rent', 'amount': 608}, {'category': 'Dining', 'amount': 96}, ...",NaN,Brandon Walker,brandon.walker2@yahoo.com,425-69-4784,10.1.102.112,M,1992-03-31,10032,78000,51,0.18,17915,False,algorithm_risk_score,NaN,NaN,NaN,NaN,NaN
2,app_215,"[{'category': 'Rent', 'amount': 109}]",NaN,Scott Moore,scott.moore94@mail.com,370-78-5178,10.240.193.250,Male,1989-10-24,10075,61000,41,0.21,37909,True,NaN,vacation,3.7,59000.0,NaN,NaN


In [6]:
summary = {
    "records": df.shape[0],
    "columns": df.shape[1],
    "missing_cells": int(df.isna().sum().sum()),
    "missing_cell_rate_pct": round(100 * df.isna().sum().sum() / (df.shape[0] * df.shape[1]), 2),
}
pd.DataFrame([summary])

,records,columns,missing_cells,missing_cell_rate_pct
0,502,21,2619,24.84


In [7]:
missing_by_col = (
    df.isna().mean()
      .sort_values(ascending=False)
      .head(15)
      .mul(100)
      .round(2)
      .reset_index()
)
missing_by_col.columns = ["column", "missing_pct"]
missing_by_col

,column,missing_pct
0,notes,99.60
1,financials.annual_salary,99.00
2,loan_purpose,90.04
3,processing_timestamp,87.65
4,decision.rejection_reason,58.17
5,decision.approved_amount,41.83
6,decision.interest_rate,41.83
7,financials.annual_income,1.00
8,applicant_info.ip_address,1.00
9,applicant_info.ssn,1.00


In [8]:
dtypes_df = df.dtypes.astype(str).value_counts().reset_index()
dtypes_df.columns = ["dtype", "count"]
dtypes_df

,dtype,count
0,object,14
1,float64,4
2,int64,2
3,bool,1


## 2. PII Identification

The first step in privacy analysis is identifying all Personally Identifiable Information (PII) in the dataset.

Examples of PII in this dataset may include:
- Full name
- Email address
- Social Security Number (SSN)
- IP address
- Date of birth

Some attributes (such as ZIP code) may act as quasi-identifiers when combined with other fields.


In [9]:
# Candidate PII / quasi-identifiers (common in credit application datasets)
PII_CANDIDATES = [
    "applicant_info.full_name",
    "applicant_info.name",
    "applicant_info.email",
    "applicant_info.ssn",
    "applicant_info.ip_address",
    "applicant_info.ip",
    "applicant_info.date_of_birth",
    "applicant_info.dob",
    "applicant_info.phone",
    "applicant_info.address",
    "applicant_info.zip_code",
    "applicant_info.postal_code",
]

# Keep only those that actually exist in this dataset
pii_cols_found = [c for c in PII_CANDIDATES if c in df.columns]
pii_cols_found

['applicant_info.full_name',
 'applicant_info.email',
 'applicant_info.ssn',
 'applicant_info.ip_address',
 'applicant_info.date_of_birth',
 'applicant_info.zip_code']

In [10]:
import re

patterns = [
    r"email",
    r"\bssn\b",
    r"ip",
    r"dob",
    r"date.*birth",
    r"birth",
    r"phone",
    r"address",
    r"full_name|fullname|name",
    r"zip|postal",
    r"passport|national_id|id_number",
]

regex = re.compile("|".join(patterns), flags=re.IGNORECASE)

pii_cols_pattern = [c for c in df.columns if regex.search(c)]
pii_cols_pattern[:50], len(pii_cols_pattern)

(['applicant_info.full_name',
  'applicant_info.email',
  'applicant_info.ssn',
  'applicant_info.ip_address',
  'applicant_info.date_of_birth',
  'applicant_info.zip_code'],
 6)

In [11]:
pii_cols = sorted(set(pii_cols_found + pii_cols_pattern))
pii_cols

['applicant_info.date_of_birth',
 'applicant_info.email',
 'applicant_info.full_name',
 'applicant_info.ip_address',
 'applicant_info.ssn',
 'applicant_info.zip_code']

In [12]:
pii_evidence = []
n = len(df)

for col in pii_cols:
    non_null = df[col].notna().sum()
    pii_evidence.append({
        "column": col,
        "non_null_rows": int(non_null),
        "non_null_pct": round(100 * non_null / n, 2),
        "missing_pct": round(100 * (1 - non_null / n), 2),
        "example_values": df[col].dropna().astype(str).head(3).tolist(),
    })

pii_evidence_df = pd.DataFrame(pii_evidence).sort_values("non_null_pct", ascending=False)
pii_evidence_df

,column,non_null_rows,non_null_pct,missing_pct,example_values
1,applicant_info.email,502,100.0,0.0,"[jerry.smith17@hotmail.com, brandon.walker2@yahoo.com, scott.moore94@mail.com]"
2,applicant_info.full_name,502,100.0,0.0,"[Jerry Smith, Brandon Walker, Scott Moore]"
0,applicant_info.date_of_birth,501,99.8,0.2,"[2001-03-09, 1992-03-31, 1989-10-24]"
5,applicant_info.zip_code,501,99.8,0.2,"[10036, 10032, 10075]"
3,applicant_info.ip_address,497,99.0,1.0,"[192.168.48.155, 10.1.102.112, 10.240.193.250]"
4,applicant_info.ssn,497,99.0,1.0,"[596-64-4340, 425-69-4784, 370-78-5178]"


In [14]:
def classify_field(col: str) -> str:
    c = col.lower()
    if "ssn" in c:
        return "PII (direct identifier)"
    if "email" in c:
        return "PII (direct identifier)"
    if "ip" in c:
        return "PII (direct identifier)"
    if "full_name" in c or c.endswith(".name") or c.endswith("name"):
        return "PII (direct identifier)"
    if "date_of_birth" in c or "dob" in c or "birth" in c:
        return "Quasi-identifier (high re-id risk)"
    if "ip_address" in c or c.endswith(".ip"):
        return "PII (direct identifier)"
    return "Review"

pii_inventory = pii_evidence_df[["column", "non_null_pct", "example_values"]].copy()
pii_inventory["category"] = pii_inventory["column"].apply(classify_field)

# Add known sensitive/protected attributes (for governance context)
SENSITIVE_ATTRS = [c for c in df.columns if c.lower().endswith("gender") or ".gender" in c.lower()]
if SENSITIVE_ATTRS:
    extra = pd.DataFrame([{
        "column": c,
        "non_null_pct": round(100 * df[c].notna().mean(), 2),
        "example_values": df[c].dropna().astype(str).head(3).tolist(),
        "category": "Sensitive / Protected attribute (fairness)"
    } for c in sorted(set(SENSITIVE_ATTRS))])
    pii_inventory = pd.concat([pii_inventory, extra], ignore_index=True)

pii_inventory.sort_values(["category", "non_null_pct"], ascending=[True, False]).reset_index(drop=True)

,column,non_null_pct,example_values,category
0,applicant_info.email,100.0,"[jerry.smith17@hotmail.com, brandon.walker2@yahoo.com, scott.moore94@mail.com]",PII (direct identifier)
1,applicant_info.full_name,100.0,"[Jerry Smith, Brandon Walker, Scott Moore]",PII (direct identifier)
2,applicant_info.zip_code,99.8,"[10036, 10032, 10075]",PII (direct identifier)
3,applicant_info.ip_address,99.0,"[192.168.48.155, 10.1.102.112, 10.240.193.250]",PII (direct identifier)
4,applicant_info.ssn,99.0,"[596-64-4340, 425-69-4784, 370-78-5178]",PII (direct identifier)
5,applicant_info.date_of_birth,99.8,"[2001-03-09, 1992-03-31, 1989-10-24]",Quasi-identifier (high re-id risk)
6,applicant_info.gender,99.8,"[Male, M, Male]",Sensitive / Protected attribute (fairness)


### Notes (PII / Quasi-identifiers / Sensitive attributes)

From the privacy audit on 502 applications, we identified multiple personal data fields:

- **Direct identifiers (PII):** full name and email are present in 100% of records; SSN and IP address are present in ~99% of records.
- **Quasi-identifiers:** date of birth and ZIP code are present in ~99.8% of records. Even if not direct identifiers, they can enable re-identification when combined with other attributes.
- **Sensitive / protected attribute (fairness):** gender is present in ~99.8% of records. While not a direct identifier, it is relevant for discrimination risk and must be handled with strict governance controls.

Implication: the raw dataset contains high-risk personal data and requires **data minimization, access control, and de-identification (pseudonymization/anonymization)** before it is used for analytics or ML training.

## 3. PII Inventory Table

We create a table that classifies each sensitive field by:

- Type of identifier
- Privacy risk level
- Necessity for model training
- Recommended treatment



In [15]:
df_priv = df.copy()
df_priv.shape

(502, 21)

In [16]:
import hashlib
import os

# In real governance: store SALT in a secret manager, not in the notebook.
SALT = os.environ.get("NOVACRED_PSEUDO_SALT", "demo_salt_change_me")

def sha256_pseudonymize(value: object, salt: str = SALT) -> object:
    if pd.isna(value):
        return value
    s = str(value).strip().lower()
    return hashlib.sha256((salt + s).encode("utf-8")).hexdigest()

# Apply to direct identifiers
COLS_TO_PSEUDONYMIZE = ["applicant_info.email", "applicant_info.ssn"]

for col in COLS_TO_PSEUDONYMIZE:
    if col in df_priv.columns:
        df_priv[col + "_pseudo"] = df_priv[col].apply(sha256_pseudonymize)

df_priv[[c for c in df_priv.columns if c in COLS_TO_PSEUDONYMIZE or c.endswith("_pseudo")]].head(5)

,applicant_info.email,applicant_info.ssn,applicant_info.email_pseudo,applicant_info.ssn_pseudo
0,jerry.smith17@hotmail.com,596-64-4340,69b1b5150350bd321de0a39f97f481bb1451388bca2bc8367ba2aa818764c293,1c1c0b2fb5736a8f5460473e4642442e1798308c3d3e75edab33f3b493b4e3d6
1,brandon.walker2@yahoo.com,425-69-4784,899d1102f79e6f42123cff33a96437a05d7dba7cac1a39c251a1f79dc0affb50,4c1c26076363896b35f2c8f2b61103140845d71da627d95d459441ddea5fef97
2,scott.moore94@mail.com,370-78-5178,5312a36f0fe9bb7dbe88dd0aca3e8e0c49ae9722b65ac2a51a7b377d3033ef4c,469d5ca1cab83724b72b9b48a635779a6969a3c5f0e77f0895728e2c864220e2
3,thomas.lee6@protonmail.com,194-35-1833,fb33d2e6ee800de1f9cf2d81ce3b6ce7064d04d94d24b49427341cf95dc67e2c,71847255a239618a588368ccef792770aa7bbb4fed36faa9f3712fbe279f9af9
4,brian.rodriguez86@aol.com,480-41-2475,3a1879e53afe9946396649e41559e56ecc35ebaac189026b3516736da9c6a1ce,9fe5ef0ab3a4e7a138e7ddebe7b7a379de1c7b44ab8d8ad49ece3ce79c7d97c4


In [17]:
checks = []
for col in COLS_TO_PSEUDONYMIZE:
    if col in df_priv.columns:
        checks.append({
            "column": col,
            "original_non_null": int(df_priv[col].notna().sum()),
            "pseudonymized_non_null": int(df_priv[col + "_pseudo"].notna().sum()),
            "unique_original": int(df_priv[col].nunique(dropna=True)),
            "unique_pseudonymized": int(df_priv[col + "_pseudo"].nunique(dropna=True)),
        })

pd.DataFrame(checks)

,column,original_non_null,pseudonymized_non_null,unique_original,unique_pseudonymized
0,applicant_info.email,502,502,494,494
1,applicant_info.ssn,497,497,494,494


In [18]:
# Create a "training-ready" view where direct identifiers are removed
drop_direct_pii = [c for c in COLS_TO_PSEUDONYMIZE if c in df_priv.columns]
df_training_ready = df_priv.drop(columns=drop_direct_pii)

df_training_ready.columns.tolist()[:15], df_training_ready.shape

(['_id',
  'spending_behavior',
  'processing_timestamp',
  'applicant_info.full_name',
  'applicant_info.ip_address',
  'applicant_info.gender',
  'applicant_info.date_of_birth',
  'applicant_info.zip_code',
  'financials.annual_income',
  'financials.credit_history_months',
  'financials.debt_to_income',
  'financials.savings_balance',
  'decision.loan_approved',
  'decision.rejection_reason',
  'loan_purpose'],
 (502, 21))

In [19]:
before_after = pd.DataFrame({
    "field": COLS_TO_PSEUDONYMIZE,
    "example_before": [df[col].dropna().astype(str).iloc[0] if col in df.columns and df[col].notna().any() else None
                       for col in COLS_TO_PSEUDONYMIZE],
    "example_after": [df_priv[col + "_pseudo"].dropna().astype(str).iloc[0] if (col + "_pseudo") in df_priv.columns and df_priv[col + "_pseudo"].notna().any() else None
                      for col in COLS_TO_PSEUDONYMIZE],
})
before_after

,field,example_before,example_after
0,applicant_info.email,jerry.smith17@hotmail.com,69b1b5150350bd321de0a39f97f481bb1451388bca2bc8367ba2aa818764c293
1,applicant_info.ssn,596-64-4340,1c1c0b2fb5736a8f5460473e4642442e1798308c3d3e75edab33f3b493b4e3d6


### Notes (Pseudonymization)

We applied **pseudonymization** to direct identifiers (email and SSN) using salted SHA-256 hashing.
This preserves analytical utility (stable unique tokens) while reducing exposure of raw identifiers.

Governance implication:
- the **salt/key must be protected** (e.g., secret manager + access control);
- downstream analytics/ML should use only the *_pseudo* fields;
- raw identifiers should be **excluded** from training datasets (data minimization).

## 4. Pseudonymization Demonstration

To reduce privacy risk, we demonstrate pseudonymization of a PII column.

Example technique:
- Hashing (SHA-256)
- Tokenization

This allows data analysis while protecting direct identifiers.


## 5. Anonymization Techniques

Some fields can be anonymized rather than pseudonymized.

Examples:
- IP address truncation
- Date of birth generalization (age bands)

These methods reduce re-identification risk.


## 6. GDPR Compliance Assessment

We map the dataset and processing pipeline to key GDPR principles:

- Lawfulness and transparency
- Data minimization
- Storage limitation
- Accountability


## 6.5 Privacy Controls Coverage 

## 7. Governance Recommendations

Based on the privacy audit, we recommend governance controls such as:

- Pseudonymization of identifiers
- Data retention policies
- Consent tracking mechanisms
- Audit logs for automated decisions
- Human oversight for high-risk decisions


## 8 EU AI Act: why this system is high-risk + governance implications 
(human oversight, logging, documentation)